# ReAcTree | Reasoning Patterns

In [9]:
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, List, Dict
from concurrent.futures import ThreadPoolExecutor
from langchain_core.globals import set_llm_cache
from langchain_community.cache import SQLiteCache
from helper import plot_mermaid, stream_invoke
import json
import re

In [10]:
# Setup Response caching
set_llm_cache(SQLiteCache(database_path=".langchain_cache.db"))

In [11]:
model = ChatOpenAI(model="gpt-4o")

In [12]:
class ReAcTreeState(TypedDict):
    problem: str
    branches: List[Dict]
    best_branch: str
    final_solution: str

NUM_BRANCHES = 3

def generate_branches(state: ReAcTreeState) -> dict:
    """Generate multiple reasoning branches/strategies."""
    response = model.invoke(
        f"Given this problem, generate {NUM_BRANCHES} DIFFERENT strategies to solve it. "
        f"Each strategy should take a fundamentally different approach.\n\n"
        f"Problem: {state['problem']}\n\n"
        f"Return JSON: [{{"
        f"\"strategy\": \"name\", \"approach\": \"description\", \"first_step\": \"action\""
        f"}}]"
    )
    cleaned = re.sub(r"```(?:json)?\s*|\s*```", "", response.content).strip()
    try:
        branches = json.loads(cleaned)
    except json.JSONDecodeError:
        branches = [{"strategy": "default", "approach": response.content, "first_step": "Analyze"}]
    return {"branches": branches}

def execute_branches(state: ReAcTreeState) -> dict:
    """Execute each branch's strategy and gather results."""
    def execute_branch(branch) -> dict:
        # Handle branch as string or dict
        if isinstance(branch, str):
            branch = {"strategy": branch, "approach": branch, "first_step": "Analyze"}
        # Execute the strategy
        response = model.invoke(
            f"You are following this strategy to solve a problem:\n\n"
            f"Problem: {state['problem']}\n"
            f"Strategy: {branch.get('strategy', 'unknown')}\n"
            f"Approach: {branch.get('approach', '')}\n"
            f"First step: {branch.get('first_step', 'Analyze')}\n\n"
            f"Execute this strategy step-by-step. Provide a detailed solution."
        )
        # Self-evaluate the branch
        eval_response = model.invoke(
            f"Rate the quality of this solution from 0.0 to 1.0.\n\n"
            f"Problem: {state['problem']}\n"
            f"Solution: {response.content[:1000]}\n\n"
            f"Return JSON: {{\"score\": 0.85, \"reason\": \"...\"}}"
        )
        eval_cleaned = re.sub(r"```(?:json)?\s*|\s*```", "", eval_response.content).strip()
        try:
            score = float(json.loads(eval_cleaned).get("score", 0.5))
        except (json.JSONDecodeError, ValueError):
            score = 0.5
        branch["result"] = response.content
        branch["score"] = score
        return branch

    with ThreadPoolExecutor() as executor:
        executed = list(executor.map(execute_branch, state["branches"]))
    return {"branches": executed}

def prune_and_deepen(state: ReAcTreeState) -> dict:
    """Prune lowest-scoring branch, then deepen top 2 with sub-branches."""
    branches = sorted(state["branches"], key=lambda b: b.get("score", 0), reverse=True)
    pruned = branches[-1]
    survivors = branches[:-1]
    print(f"[Prune] Dropping '{pruned.get('strategy', '?')}' (score={pruned.get('score', 'N/A')})")
    for b in survivors:
        b["depth"] = 1
    # Deepen: each surviving branch generates one sub-branch at depth 2
    def deepen_branch(branch):
        response = model.invoke(
            f"You previously solved this problem with strategy '{branch.get('strategy', '')}'.\n"
            f"Your solution so far:\n{branch.get('result', '')[:500]}\n\n"
            f"Problem: {state['problem']}\n\n"
            f"Now go ONE LEVEL DEEPER: refine or extend your solution. "
            f"Add implementation details, handle edge cases, or optimize."
        )
        eval_resp = model.invoke(
            f"Rate this refined solution from 0.0 to 1.0.\n\n"
            f"Problem: {state['problem']}\nSolution: {response.content[:800]}\n\n"
            f'Return JSON: {{"score": 0.85}}'
        )
        cleaned = re.sub(r"```(?:json)?\s*|\s*```", "", eval_resp.content).strip()
        try:
            score = float(json.loads(cleaned).get("score", 0.5))
        except (json.JSONDecodeError, ValueError):
            score = 0.5
        return {**branch, "result": response.content, "score": score, "depth": 2}
    with ThreadPoolExecutor() as executor:
        deepened = list(executor.map(deepen_branch, survivors))
    # Print tree structure
    print("\n[Tree Structure]")
    for b in state["branches"]:
        marker = "X (pruned)" if b is pruned else f"score={b.get('score', 'N/A')}"
        print(f"  depth=1: {b.get('strategy', '?')} [{marker}]")
    for b in deepened:
        print(f"    depth=2: {b.get('strategy', '?')} [score={b.get('score', 'N/A')}]")
    return {"branches": deepened}

def finalize(state: ReAcTreeState) -> dict:
    """Produce final solution from the best branch (after pruning + deepening)."""
    if not state["branches"]:
        return {"final_solution": "No viable branches found."}
    best = max(state["branches"], key=lambda b: b.get("score", 0))
    branch_summary = "\n".join(
        f"  depth={b.get('depth', 1)} {b.get('strategy', 'unknown')}: score={b.get('score', 'N/A')}"
        for b in state["branches"]
    )
    return {
        "final_solution": (
            f"Selected Strategy: {best.get('strategy', 'unknown')} "
            f"(depth={best.get('depth', 1)}, score={best.get('score', 'N/A')})\n\n"
            f"All branches after pruning + deepening:\n{branch_summary}\n\n"
            f"Solution:\n{best.get('result', 'No result')}"
        )
    }

In [13]:
graph = StateGraph(ReAcTreeState)
graph.add_sequence([("branch", generate_branches), ("execute", execute_branches), ("prune_deepen", prune_and_deepen), ("finalize", finalize)])
graph.add_edge(START, "branch")
graph.add_edge("finalize", END)

reactree = graph.compile()

In [14]:
# Plot the workflow
plot_mermaid(reactree)

```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	branch(branch)
	execute(execute)
	prune_deepen(prune_deepen)
	finalize(finalize)
	__end__([<p>__end__</p>]):::last
	__start__ --> branch;
	branch --> execute;
	execute --> prune_deepen;
	prune_deepen --> finalize;
	finalize --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

```

In [15]:
result = reactree.invoke({
    "problem": "Design a system to detect and prevent fraud in real-time for a payment processing platform "
               "handling 10,000 transactions per second."
})
print(result["final_solution"])

[Prune] Dropping 'Blockchain for Immutable Logging and Auditing' (score=0.5)

[Tree Structure]
  depth=1: Machine Learning Anomaly Detection [score=0.85]
  depth=1: Rule-Based System with Behavioral Profiling [score=0.65]
  depth=1: Blockchain for Immutable Logging and Auditing [X (pruned)]
    depth=2: Machine Learning Anomaly Detection [score=0.5]
    depth=2: Rule-Based System with Behavioral Profiling [score=0.85]
Selected Strategy: Rule-Based System with Behavioral Profiling (depth=2, score=0.85)

All branches after pruning + deepening:
  depth=2 Machine Learning Anomaly Detection: score=0.5
  depth=2 Rule-Based System with Behavioral Profiling: score=0.85

Solution:
To refine and extend the solution for designing a real-time fraud detection and prevention system capable of handling 10,000 transactions per second, we need to delve deeper into architecture, optimization, and implementation details.

### Step 1: System Architecture and Design

#### 1.1 Scalable Infrastructure
- **Cl

In [16]:
stream_invoke(reactree, {
    "problem": "Design a system to detect and prevent fraud in real-time for a payment processing platform "
               "handling 10,000 transactions per second."
})


────────────────────────────────────────────────────────────────────────────────
  STREAMING EXECUTION
────────────────────────────────────────────────────────────────────────────────

[Prune] Dropping 'Blockchain for Immutable Logging and Auditing' (score=0.5)

[Tree Structure]
  depth=1: Machine Learning Anomaly Detection [score=0.85]
  depth=1: Rule-Based System with Behavioral Profiling [score=0.65]
  depth=1: Blockchain for Immutable Logging and Auditing [X (pruned)]
    depth=2: Machine Learning Anomaly Detection [score=0.5]
    depth=2: Rule-Based System with Behavioral Profiling [score=0.85]
────────────────────────────────────────────────────────────────────────────────
  EXECUTION COMPLETE
────────────────────────────────────────────────────────────────────────────────



{'problem': 'Design a system to detect and prevent fraud in real-time for a payment processing platform handling 10,000 transactions per second.',
 'branches': [{'strategy': 'Machine Learning Anomaly Detection',
   'approach': 'Leverage machine learning models to identify anomalous transactions in real-time. Train the models using historical transaction data, labeling known fraudulent and legitimate activities. Deploy these models to continuously score transactions for fraud likelihood.',
   'first_step': 'Collect and preprocess historical transaction data for model training, ensuring it includes labels for both fraudulent and legitimate transactions.',
   'result': "To design a fraud detection system capable of handling 10,000 transactions per second in real-time, we need to delve deeper into specific aspects of the system such as data architecture, model choice and optimization, deployment strategies, and continuous improvement mechanisms. Here's a more detailed approach:\n\n### Step